In [1]:
import os
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import xarray as xr
import yaml
import gc

import cdsapi

In [2]:
# Configuración de fechas y rutas
START_DATE = "2026-01-20"  # Fecha de inicio en formato YYYY-MM-DD
END_DATE = "2026-02-10"    # Fecha de fin en formato YYYY-MM-DD

# Parsear fechas
start = pd.to_datetime(START_DATE)
end = pd.to_datetime(END_DATE)

# Generar rango de fechas diario
datelist = pd.date_range(START_DATE, END_DATE, freq='D')

print(f"Período: {START_DATE} a {END_DATE}")
print(f"Número de días a procesar: {len(datelist)}")

# Rutas generales
target_dir = "."
preprocessed_dir = "./preprocessed_data"
output_dir = "./output_data"
yaml_config_name = "workflow_config.yaml"

# Configuración de descarga
skip_exist = True
area = [80, -180, -20, 0]  # None para global, o [N, W, S, E]
grid = [0.1, 0.1]

# Variables a descargar
ml_variables = {
    "u": 131,
    "v": 132,
    "q": 133,
}

surface_variables = {
    "tp": "total_precipitation",
    "e": "evaporation",
    "sp": "surface_pressure",
    "tcw": "total_column_water",
}

# Niveles de modelo
levels = "20/40/60/80/90/95/100/105/110/115/120/123/125/128/130/131/132/133/134/135/136/137"

# Tiempos (24 horas)
times = [f"{h:02d}:00" for h in range(24)]

print(f"\nConfiguraciones:")
print(f"- Directorio de salida preprocessado: {preprocessed_dir}")
print(f"- Directorio de output: {output_dir}")
print(f"- Variables ML: {list(ml_variables.keys())}")
print(f"- Variables superficie: {list(surface_variables.keys())}")

Período: 2026-01-20 a 2026-02-10
Número de días a procesar: 22

Configuraciones:
- Directorio de salida preprocessado: ./preprocessed_data
- Directorio de output: ./output_data
- Variables ML: ['u', 'v', 'q']
- Variables superficie: ['tp', 'e', 'sp', 'tcw']


In [ ]:
print("="*80)
print("INICIANDO WORKFLOW DIARIO COMPLETO")
print("="*80)

# Inicializar cliente CDS
c = cdsapi.Client(timeout=600, quiet=False)

# Función de preprocesamiento
def procesar_dia(current_date):
    """Procesa los archivos de un día específico"""
    year = current_date.year
    month = current_date.month
    day = current_date.day
    month_str = f"{month:02d}"
    day_str = f"{day:02d}"
    date_str = current_date.strftime("%Y-%m-%d")
    
    dia_path = Path(target_dir) / f"{year}" / month_str / day_str
    
    if not dia_path.exists():
        print(f"✗ Carpeta {dia_path} no encontrada para {date_str}.")
        return
    
    # Crear directorio de preprocesamiento
    output_base_dir = Path(preprocessed_dir) / "descompuestos"
    output_base_dir.mkdir(exist_ok=True, parents=True)
    
    # Crear carpeta de salida para el mes
    output_dir = output_base_dir / month_str
    output_dir.mkdir(exist_ok=True, parents=True)
    
    print(f"  Preprocesando día {date_str}...")
    
    # Variables a procesar
    surface_vars = list(surface_variables.keys())
    ml_vars = list(ml_variables.keys())
    
    # Procesar variables de superficie
    for var in surface_vars:
        archivo_diario = dia_path / f"ERA5_{date_str}_{var}.nc"
        if not archivo_diario.exists():
            print(f"    ℹ Archivo {archivo_diario.name} no encontrado, saltando.")
            continue
        
        try:
            with xr.open_dataset(archivo_diario, chunks={'time': 1}, decode_cf=True) as ds:
                ds = ds.drop_vars(['expver', 'number'], errors='ignore')
                if 'ps' in ds.data_vars:
                    ds = ds.rename({'ps': 'sp'})
                if 'valid_time' in ds.coords:
                    ds = ds.rename({'valid_time': 'time'})
                
                # Convertir tipos
                for var_name in ds.data_vars:
                    ds[var_name] = ds[var_name].astype(np.float64)
                
                # Transponer
                ds = ds.transpose('time', 'latitude', 'longitude')
                
                output_filename = f"ERA5_{date_str}_{var}.nc"
                output_path = output_dir / output_filename
                ds.to_netcdf(output_path)
                
        except Exception as e:
            print(f"    ✗ Error procesando {var} para {date_str}: {e}")
    
    # Procesar variables de modelo (ML)
    for ml in ml_vars:
        archivo_diario = dia_path / f"ERA5_{date_str}_ml_{ml}.nc"
        if not archivo_diario.exists():
            print(f"    ℹ Archivo {archivo_diario.name} no encontrado, saltando.")
            continue
        
        try:
            with xr.open_dataset(archivo_diario, chunks={'time': 1}, decode_cf=True) as ds:
                ds = ds.drop_vars(['expver', 'number'], errors='ignore')
                if 'valid_time' in ds.coords:
                    ds = ds.rename({'valid_time': 'time'})
                if 'ps' in ds.data_vars:
                    ds = ds.rename({'ps': 'sp'})
                if 'model_level' in ds.dims:
                    ds = ds.rename({'model_level': 'level'})
                
                # Convertir tiempo si es cftime
                if hasattr(ds['time'].values[0], 'calendar'):
                    ds['time'] = pd.to_datetime(ds['time'].values)
                
                # Convertir tipos
                for var_name in ds.data_vars:
                    ds[var_name] = ds[var_name].astype(np.float64)
                
                # Transponer
                ds = ds.transpose('time', 'level', 'latitude', 'longitude')
                ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180))
                
                output_filename = f"ERA5_{date_str}_ml_{ml}.nc"
                output_path = output_dir / output_filename
                ds.to_netcdf(output_path)
                
        except Exception as e:
            print(f"    ✗ Error procesando ml_{ml} para {date_str}: {e}")

# Función para crear YAML
def crear_yaml_config(start_date, end_date, yaml_path):
    """Crea archivo de configuración YAML para un rango de fechas"""
    yaml_text = f"""# Output
preprocessed_data_folder: {preprocessed_dir}
output_folder: {output_dir}
output_frequency: "1d"

# Preprocessing
filename_template: "{'preprocessed_data/descompuestos/{month:02}/ERA5_{year}-{month:02d}-{day:02d}{levtype}_{variable}.nc' }"
preprocess_start_date: "{start_date.strftime('%Y-%m-%dT00:00')}"
preprocess_end_date: "{end_date.strftime('%Y-%m-%dT23:00')}"
level_type: model_levels
levels: [20,40,60,80,90,95,100,105,110,115,120,123,125,128,130,131,132,133,134,135,136,137]
input_frequency: "1h"

# Tracking
tracking_direction: backward
tracking_domain: null
periodic_boundary: False
tracking_start_date: "{start_date.strftime('%Y-%m-%dT00:00')}"
tracking_end_date: "{end_date.strftime('%Y-%m-%dT23:00')}"
timestep: 600
kvf: 3

# Tagging
tagging_start_date: "{start_date.strftime('%Y-%m-%dT00:00')}"
tagging_end_date: "{end_date.strftime('%Y-%m-%dT23:00')}"
restart: False
tagging_region: shapes/sinu.nc
target_frequency: "10min"
"""

    with open(yaml_path, 'w') as f:
        f.write(yaml_text)

# Función para ejecutar WAM2Layers
def ejecutar_wam2layers(yaml_path):
    """Ejecuta WAM2Layers con el archivo YAML especificado"""
    wam2layers_cmd = ["wam2layers", "preprocess", "era5", str(yaml_path)]
    
    try:
        result = subprocess.run(
            wam2layers_cmd,
            capture_output=True,
            text=True,
            check=False
        )
        
        if result.returncode != 0:
            print(f"    ✗ wam2layers error: {result.returncode}")
            return False
        else:
            return True
            
    except Exception as e:
        print(f"    ✗ Error ejecutando wam2layers: {e}")
        return False

# Loop diario
for current_date in datelist:
    year = current_date.year
    month = current_date.month
    day = current_date.day
    month_str = f"{month:02d}"
    day_str = f"{day:02d}"
    date_str = current_date.strftime("%Y-%m-%d")
    
    print(f"\n--- Día {date_str} ---")
    
    outfolder = Path(target_dir) / f"{year}" / f"{month_str}" / f"{day_str}"
    outfolder.mkdir(exist_ok=True, parents=True)
    
    # Descarga
    print(f"  Descargando datos...")
    download_success = True
    
    # Descargar variables de superficie
    for variable, long_name in surface_variables.items():
        outfile = f"ERA5_{date_str}_{variable}.nc"
        filepath = outfolder / outfile
        
        if filepath.exists() and skip_exist:
            print(f"    ✓ {filepath} ya existe")
        else:
            try:
                c.retrieve(
                    "reanalysis-era5-single-levels",
                    {
                        "product_type": "reanalysis",
                        "variable": long_name,
                        "year": f"{year}",
                        "month": month_str,
                        "day": day_str,
                        "time": times,
                        "grid": grid,
                        "format": "netcdf",
                    },
                    str(filepath),
                )
                print(f"    ✓ {variable} descargado")
            except Exception as e:
                print(f"    ✗ Error descargando {variable}: {e}")
                download_success = False
    
    # Descargar variables de modelo
    for variable, param in ml_variables.items():
        outfile = f"ERA5_{date_str}_ml_{variable}.nc"
        filepath = outfolder / outfile
        
        if filepath.exists() and skip_exist:
            print(f"    ✓ {filepath} ya existe")
        else:
            try:
                c.retrieve(
                    "reanalysis-era5-complete",
                    {
                        "time": times,
                        "dates": date_str,
                        "levelist": levels,
                        "param": param,
                        "class": "ea",
                        "expver": "1",
                        "levtype": "ml",
                        "stream": "oper",
                        "type": "an",
                        "format": "netcdf",
                        "grid": grid,
                    },
                    str(filepath),
                )
                print(f"    ✓ ml_{variable} descargado")
            except Exception as e:
                print(f"    ✗ Error descargando ml_{variable}: {e}")
                download_success = False
    
    if not download_success:
        print(f"  ✗ Saltando día {date_str} por errores en descarga")
        continue
    
    # Preprocesamiento
    procesar_dia(current_date)
    
    # Crear YAML
    yaml_path = Path(f"workflow_config_{date_str}.yaml")
    crear_yaml_config(current_date, current_date, yaml_path)
    
    # Ejecutar WAM2Layers
    print(f"  Ejecutando WAM2Layers...")
    if ejecutar_wam2layers(yaml_path):
        print(f"  ✓ Día {date_str} completado")
    else:
        print(f"  ✗ Error en ejecución para {date_str}")

print("\n" + "="*80)
print("✓ WORKFLOW DIARIO COMPLETADO")
print("="*80)